# CLOB simulation convergence summary

This notebook summarizes the saved run-level results in `sim_output/clob_simulation_summary.csv`; it does not rerun simulations. Sustained convergence means the price stayed within the configured tolerance of the fundamental probability for the configured number of consecutive periods.

In [ ]:
from pathlib import Path

import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import PercentFormatter

csv_path = Path("sim_output/clob_simulation_summary.csv")
if not csv_path.is_file():
    raise FileNotFoundError(
        f"Could not find {csv_path.resolve()}. Run this notebook from the repository root, "
        "which contains sim_output/clob_simulation_summary.csv."
    )

summary = pd.read_csv(csv_path)
required_columns = {
    "run", "converged", "first_passage_time", "confirmation_time",
    "dimensionless_convergence_time",
}
missing_columns = required_columns.difference(summary.columns)
if missing_columns:
    raise ValueError(
        f"{csv_path} is missing required columns: {sorted(missing_columns)}"
    )

summary["converged"] = summary["converged"].astype("boolean")
run_count = len(summary)
converged_count = int(summary["converged"].fillna(False).sum())
not_converged_count = run_count - converged_count
convergence_rate = converged_count / run_count if run_count else float("nan")

print(f"Simulation runs: {run_count}")
print(f"Converged: {converged_count} ({convergence_rate:.1%})")
print(f"Not converged within the simulation: {not_converged_count}")

## Convergence by run

`first_passage_time` is the start of the first qualifying window; `confirmation_time` is when that window is complete. Non-converged runs are censored at the end of the simulation.

In [ ]:
convergence_by_run = summary[
    [
        "run",
        "converged",
        "first_passage_time",
        "confirmation_time",
        "dimensionless_convergence_time",
    ]
].sort_values("run", ignore_index=True)
display(convergence_by_run)

## Convergence-time summary

Time summaries below include only runs that converged; non-converged runs are not counted as if they converged at the simulation endpoint.

In [ ]:
converged_runs = summary.loc[summary["converged"].fillna(False)]
if converged_runs.empty:
    print("No runs converged; convergence-time statistics are unavailable.")
else:
    first_passage = converged_runs["first_passage_time"].dropna()
    confirmation = converged_runs["confirmation_time"].dropna()
    dimensionless = converged_runs["dimensionless_convergence_time"].dropna()
    print(f"Converged runs: {len(converged_runs)}")
    if not first_passage.empty:
        print(
            "First passage (periods): "
            f"median={first_passage.median():.2f}, mean={first_passage.mean():.2f}"
        )
    if not confirmation.empty:
        print(
            "Confirmation (periods): "
            f"median={confirmation.median():.2f}, mean={confirmation.mean():.2f}"
        )
    if not dimensionless.empty:
        print(
            "Dimensionless first-passage time: "
            f"median={dimensionless.median():.3f}, mean={dimensionless.mean():.3f}"
        )
    else:
        print("Dimensionless convergence times are unavailable in the saved results.")

## Time to sustained convergence

This figure shows the Kaplan–Meier estimate of the cumulative probability of first sustained convergence, with pointwise 95% log-log confidence intervals. Non-converged runs are right-censored at the simulation horizon. Event time is the start of the first qualifying window; confirmation occurs one window later. The saved CSV does not store the simulation horizon, so `SIMULATION_HORIZON` below must match `T_ELECTION` used to generate these results.

In [ ]:
SIMULATION_HORIZON = 100  # T_ELECTION for the saved summary CSV
event_observed = summary["converged"].fillna(False).astype(bool)
event_times = pd.to_numeric(
    summary.loc[event_observed, "first_passage_time"], errors="coerce"
)
if event_times.isna().any():
    raise ValueError("Converged runs must have a valid first_passage_time.")
if (event_times > SIMULATION_HORIZON).any():
    raise ValueError("A convergence time exceeds SIMULATION_HORIZON.")

observed_times = [
    float(event_time) if observed else float(SIMULATION_HORIZON)
    for event_time, observed in zip(
        summary["first_passage_time"], event_observed
    )
]
event_grid = sorted(event_times.unique())

survival = 1.0
greenwood_sum = 0.0
plot_times = [0.0]
cumulative_convergence = [0.0]
ci_lower = [0.0]
ci_upper = [0.0]
z_975 = 1.959963984540054

for event_time in event_grid:
    at_risk = sum(time >= event_time for time in observed_times)
    events = int((event_times == event_time).sum())
    survival *= 1.0 - events / at_risk

    if events == at_risk:
        greenwood_sum = math.inf
    elif math.isfinite(greenwood_sum):
        greenwood_sum += events / (at_risk * (at_risk - events))

    if survival <= 0.0:
        survival_lower, survival_upper = 0.0, 0.0
    elif survival >= 1.0 or greenwood_sum == 0.0:
        survival_lower = survival_upper = survival
    else:
        log_neg_log_survival = math.log(-math.log(survival))
        standard_error = math.sqrt(greenwood_sum) / abs(math.log(survival))
        survival_lower = math.exp(
            -math.exp(log_neg_log_survival + z_975 * standard_error)
        )
        survival_upper = math.exp(
            -math.exp(log_neg_log_survival - z_975 * standard_error)
        )

    plot_times.append(float(event_time))
    cumulative_convergence.append(1.0 - survival)
    ci_lower.append(1.0 - survival_upper)
    ci_upper.append(1.0 - survival_lower)

if plot_times[-1] < SIMULATION_HORIZON:
    plot_times.append(float(SIMULATION_HORIZON))
    cumulative_convergence.append(cumulative_convergence[-1])
    ci_lower.append(ci_lower[-1])
    ci_upper.append(ci_upper[-1])

fig, ax = plt.subplots(figsize=(6.4, 4.2), constrained_layout=True)
ax.fill_between(
    plot_times, ci_lower, ci_upper, step="post", color="#4C78A8", alpha=0.20,
    linewidth=0, label="95% pointwise confidence interval",
)
ax.step(
    plot_times, cumulative_convergence, where="post", color="#234F7D",
    linewidth=2.0, label="Estimated cumulative convergence probability",
)

censored_count = int((~event_observed).sum())
if censored_count:
    ax.scatter(
        [SIMULATION_HORIZON], [cumulative_convergence[-1]], marker="+",
        s=55, linewidths=1.4, color="#333333",
        label=f"Right-censored at horizon (n={censored_count}; marks overlap)",
        zorder=3,
    )

ax.set(
    xlim=(0, SIMULATION_HORIZON),
    ylim=(0, max(0.20, min(1.0, ci_upper[-1] * 1.15))),
    xlabel="Periods to first sustained convergence",
    ylabel="Cumulative probability of convergence",
)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
ax.grid(axis="y", color="#D9D9D9", linewidth=0.7)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False, loc="upper left", fontsize=8)
ax.set_title(
    f"Time to sustained convergence ({converged_count}/{run_count} runs converged)",
    fontsize=11,
)

fig.savefig("figures/convergence_time_km.png", dpi=600, bbox_inches="tight")
fig.savefig("figures/convergence_time_km.pdf", bbox_inches="tight")
plt.show()
print("Saved convergence_time_km.png and convergence_time_km.pdf")